# LAB | First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Learning goal:** build one retrieval path over two sources and **prove** every answer is grounded in evidence you can inspect.

**Your sources**
- `podcast.txt`: a podcast transcript about the EU *Ethics Guidelines for Trustworthy AI* (2019)
- `eu_ai_act.pdf`: Regulation (EU) 2024/1689, the AI Act (144 pages)

**What you submit:** a `lab_proof.md` containing your code (or a link to this notebook), **one query**, the **retrieved evidence**, the **final answer**, and **one failure or limitation**. There's a template at the end of this notebook.

---

### How this notebook works

| Label | Meaning |
|---|---|
| **PROVIDED** | Code is given. Run it and read it. |
| **YOUR TURN** | Replace the `# TODO` lines with your own code. |
| **OPTIONAL (Advanced)** | Do this if you have time; it's not required to pass. |

Each step ends with a **Done when** check. Don't move on until it passes.

### Prompt discipline
You may use AI to **review, debug or critique** your work, but not to write your final answer or your `lab_proof.md`. A good review prompt:

> *Role: Retrieval reviewer. Task: Review my plan for grounding and verification. Inputs: my sources, query, retrieval path, and evidence. Constraints: Do not produce the final answer. Output: three grounding risks, two trace checks, one simplification.*

---
## Step 0: Environment [PROVIDED]

1. Create a `.env` file next to this notebook containing `OPENAI_API_KEY=...` and `PINECONE_API_KEY=...`
2. Create a folder called `data/` next to this notebook and put `podcast.txt` and `eu_ai_act.pdf` inside it.
3. Run the two cells below.

**Warning:** Never paste API keys into a notebook cell, and never commit your `.env` file.

In [ ]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

---
## Step 1: Setup & Data Preparation [PROVIDED, except 1d]

**Goal:** turn two very different sources into chunks that each carry **metadata**, meaning labels that say *where the chunk came from*.

**Why metadata matters:** when the system answers *"Fines are up to €35M"*, a reviewer must be able to trace that sentence back to *"EU AI Act, page 115"*. Without metadata you can't trace an answer, and you can't filter results either (Step 5).

Parts 1a–1c are done for you: run each cell and **read the output**. Part 1d (embeddings) is yours to write.

### 1a. Load both sources

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

### 1b. Chunk with metadata

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",  # unique id -> lets a reviewer trace an exact chunk
        }
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

### 1c. Inspect your chunks (always look at your data!)

In [ ]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

### 1d. Embeddings model [YOUR TURN]

Set up the embedding model yourself and test it on **one** chunk.

**What to do**
1. Import `OpenAIEmbeddings` from `langchain_openai`.
2. Create a variable called `embeddings` using the model `text-embedding-3-small`. Step 2 uses this exact variable name.
3. Embed the text of `all_chunks[0]` with `embeddings.embed_query(...)` and print the length of the vector.

**Hint:** see **Section 2 of `rag_intro.ipynb`**.
**Note:** you don't embed **all** chunks here. That happens in Step 2, when you upload them to Pinecone, so you only pay for them once.

In [ ]:
# YOUR TURN: set up the embedding model and test it on one chunk

# TODO: import OpenAIEmbeddings

# TODO: embeddings = ...

# TODO: embed all_chunks[0].page_content and print the vector length

### Stop and think (write your answers in `lab_proof.md` later)

1. Our metadata has `source_type`, `source_name`, `chunk_id` and `page`. **What's missing?** Think about a user asking *"What does Article 5 say?"* or *"Only show me recitals."* Could you filter on that today?
2. The podcast has no pages. How would a reviewer find where a podcast chunk came from?

You'll come back to this in **Step 5**.

**Done when:** you can see chunk counts for both sources, every chunk you print shows a `metadata` dict, and your test embedding has length **1536**.

---
## Step 2: Embeddings + Baseline Retrieval [YOUR TURN]

**Goal:** build a plain vector search. This is your **baseline**: every later step must beat it, or it isn't worth the extra cost.

**What to do**
1. Create (or connect to) a Pinecone index. Use dimension `1536` and metric `cosine`.
2. Upload `all_chunks` with `PineconeVectorStore.from_documents(...)`. This is the step that embeds every chunk.
3. Write `baseline_search(query, k)`. It should return `(doc, score)` pairs using `similarity_search_with_score`.
4. Run it on the **fixed test queries** below and save the results.

**Hint:** Stuck? Look at **Section 3–4 of `rag_intro.ipynb`** (yesterday's notebook).
**Warning:** Only run the upload cell **once**. Running it again uploads duplicate chunks.

In [ ]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# PROVIDED: helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [ ]:
# YOUR TURN: create / connect to the Pinecone index
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"   # use lowercase letters and dashes only

# TODO: if index_name does not exist yet, create it (dimension=1536, metric="cosine",
#       spec=ServerlessSpec(cloud="aws", region="us-east-1"))

# TODO: vectorstore = PineconeVectorStore.from_documents(...)

In [ ]:
# YOUR TURN: baseline retrieval
def baseline_search(query, k=5):
    # TODO: return vectorstore.similarity_search_with_score(...)
    pass

baseline_results = {}
for q in TEST_QUERIES:
    # TODO: call baseline_search, store the result in baseline_results[q], and print it with show_results
    pass

**Done when:** each test query prints 5 results with a score, a source type and a page or chunk id.

**Note it down:** for which query does the **top result** look wrong or only half-relevant? That's your *first match ≠ right match* case.

---
## Step 3: LLM Relevance Scoring [OPTIONAL (Advanced)]

**Goal:** have an LLM judge how relevant each retrieved chunk is, then re-order the results.

**What to do**
1. Retrieve a wider set of candidates (e.g. `k=10`) with `baseline_search`.
2. For each chunk, ask `gpt-4o-mini` to score its relevance to the query from 0 to 1.
3. Combine the two scores, e.g. `combined = 0.3 * similarity + 0.7 * llm_score`, and sort by `combined`.
4. Compare the new top 3 with the baseline top 3.

**Hint:** Reuse `rerank_documents()` from **Section 3 of `rag_pipeline.ipynb`**. It already parses the score and prints it.
**Warning:** This makes 1 LLM call **per chunk, per query**. Count the calls you make.

In [ ]:
# YOUR TURN
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_relevance_score(query, doc):
    # TODO: prompt the LLM for a 0-1 score, parse it robustly, return a float
    pass

def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7):
    # TODO: retrieve k_initial -> score each chunk -> combine scores -> sort -> return top k_final as (doc, combined_score)
    pass

**Done when:** you can show at least one query where the order changed, and say whether it got better or worse.

---
## Step 4: Dedicated Reranker (Cross-Encoder or Cohere) [OPTIONAL (Advanced)]

**Goal:** use a model built specifically for reranking. It's usually faster and cheaper than an LLM, and often better.

**Option A: local cross-encoder (no API key needed)**
```
%pip install -q sentence-transformers
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, doc.page_content) for doc in docs])
```

**Option B: Cohere Rerank API** (needs `COHERE_API_KEY` in `.env`; see the Cohere Rerank docs)

**What to do:** rerank the same 10 baseline candidates, keep the top 5, and compare with Step 3.

In [ ]:
# YOUR TURN
def crossencoder_rerank(query, k_initial=10, k_final=5):
    # TODO: retrieve k_initial candidates -> score (query, chunk) pairs -> sort -> return top k_final as (doc, score)
    pass

**Done when:** you have a side-by-side of baseline vs reranked top 3 for at least two queries.

---
## Step 5: Metadata Filtering [YOUR TURN]

**Goal:** restrict search to the right part of the knowledge base *before* ranking.

**What to do**
1. Filter by source using Pinecone's filter syntax:
   `vectorstore.similarity_search_with_score(query, k=5, filter={"source_type": {"$eq": "regulation"}})`
2. Run one query twice: once with **only the regulation**, once with **only the podcast**. How do the answers differ?
3. **The real lesson:** try to filter on something your metadata *doesn't* have yet, such as `section` (`"recital"` / `"article"` / `"annex"`) or `article` (`"Article 5"`). You can't.
   → Go back to `chunk_with_metadata` in Step 1, add the field, **delete the index and re-upload** (`pc.delete_index(index_name)`), then try again.

**Hint:** Hints for the EU AI Act: in this PDF the recitals run up to about page 44, the Articles start at *"CHAPTER I"*, and the Annexes start at *"ANNEX I"*. A regex like `r"^Article\s+(\d+)"` on each line can detect article headings.
**Warning:** Pinecone metadata values can't be `None`. Leave the key out instead.

In [ ]:
# YOUR TURN: filter by source_type
# TODO

In [ ]:
# YOUR TURN: filter by the NEW metadata field you added (section / article / ...)
# TODO

**Done when:** you can run a query that only returns, e.g., *Article* chunks from the EU AI Act, and you can explain why you had to rebuild the index to do it.

---
## Step 6: Complete RAG Pipeline with Reranking [YOUR TURN]

**Goal:** plug your best retrieval method into a full RAG chain that **cites its sources**.

**What to do**
1. Build an LCEL chain: `retrieve → format context → prompt → LLM → answer` (see **Section 5 of `rag_intro.ipynb`**).
2. In `format_context`, put each chunk's source and page **inside** the context, e.g. `[EU AI Act, p.51]`, and tell the LLM to cite them.
3. Make it easy to switch retrieval methods: baseline vs. reranked should be a **1–2 line change**.
4. Make the model say *"I don't know based on the sources"* when the context doesn't contain the answer.

In [ ]:
# YOUR TURN
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def format_context(results):
    # TODO: join chunks into one string, each prefixed with its source + page/chunk_id
    pass

rag_prompt = ChatPromptTemplate.from_template(
    # TODO: write a prompt that answers ONLY from context, cites [source, page], and admits when it doesn't know
    ""
)

def answer(query, retrieve_fn=baseline_search):   # <- swap retrieve_fn to switch methods
    # TODO: retrieve -> format -> prompt | llm | StrOutputParser() -> return answer AND the results used
    pass

**Done when:** one query gives an answer with citations, and you can point to the exact chunk behind each claim.

---
## Step 7: Evaluate: With vs. Without Reranking [YOUR TURN]

**Goal:** decide, with evidence, whether reranking was worth it.

**What to do:** for each test query, read the answers and the retrieved chunks yourself, then fill in the table. You are the judge: no automatic metric.

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 | Baseline | | | | |
| Q1 | Reranked | | | | |
| Q2 | Baseline | | | | |
| Q2 | Reranked | | | | |
| … | | | | | |

**Then answer:**
- Did reranking change the answer, or only the order?
- What did it cost (extra calls, seconds per query)? Would a client pay for that?

In [ ]:
# YOUR TURN (optional helper): print baseline vs reranked answers side by side, and time each one
import time

for q in TEST_QUERIES:
    # TODO: time answer(q, baseline_search) and answer(q, <your reranker>), then print both
    pass

---
## Step 8: Write `lab_proof.md` [YOUR TURN]

Create `lab_proof.md` in your repo (this is the **entry point for your reviewer**). Use this template:

```markdown
# Lab Proof: First match ≠ right match

## 1. What I built
One or two sentences, plus a link to this notebook.

## 2. Query
"..."

## 3. Retrieved evidence
| Rank | Source | Page / chunk_id | Score | Snippet |
|---|---|---|---|---|

## 4. Final answer
(paste the answer, including its citations)

## 5. Trace
Which claim in the answer comes from which chunk?

## 6. Baseline vs reranked
What changed? Was it worth the cost?

## 7. Failure or limitation
Where could this system give a plausible but unsupported answer?
```

**Before you submit:**
- [ ] Only files for this lab are in the repo
- [ ] No API keys, `.env` or tokens are committed
- [ ] `lab_proof.md` is the entry point
- [ ] A reviewer can trace **one** answer back to the exact retrieved chunk

---
### Extensions (optional)
- **Hybrid search:** combine BM25 keyword search with vector search, then rerank
- **Cost analysis:** cost per query for baseline vs LLM scoring vs cross-encoder
- **Failure test:** ask something neither source covers. Does your pipeline say "I don't know"?